# 02｜构建 ViT feature store 与滚动切分

特征覆盖 `feature_valid & in_universe_t`。无标签样本 target 保持 NaN，并由 `label_valid.npy` 标记；训练/验证仅使用有效标签，预测覆盖全部 T 日有效 Universe 特征。

## 步骤 1：设置数据构建参数

In [ ]:
from pathlib import Path
import os

CURRENT_DIR = Path.cwd().resolve()
VIT_DIR = CURRENT_DIR if (CURRENT_DIR / 'vit_stage04_feature_store.py').exists() else CURRENT_DIR / 'ViT'
FEATURE_STAGE_PATH = VIT_DIR / 'vit_stage04_feature_store.py'
os.environ['VIT_FEATURE_STORE_OVERWRITE'] = '0'
os.environ.pop('VIT_MAX_FEATURE_DATES', None)
print({'ViT目录': str(VIT_DIR), '覆盖特征库': False, '日期上限': None})

## 步骤 2：构建 feature store 和年度索引

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(FEATURE_STAGE_PATH)], cwd=VIT_DIR, check=True)

## 步骤 3：核对 NaN target 与滚动样本数

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

store = VIT_DIR / 'data/feature_store'
manifest = json.loads((store / 'dataset_manifest.json').read_text(encoding='utf-8'))
targets = np.load(store / 'targets.npy', mmap_mode='r')
label_valid = np.load(store / 'label_valid.npy', mmap_mode='r')
assert np.array_equal(np.isfinite(targets), label_valid)
display(pd.Series({'数据签名': manifest['signature'], '特征样本': manifest['sample_count'], '有效标签': manifest['label_valid_count'], '无标签但可预测': int((~label_valid).sum())}, name='value').to_frame())
display(pd.read_parquet(store / 'rolling_split_summary.parquet'))